# GeoSalud Cotopaxi — Table 1 verification

**Purpose.** This notebook reproduces the annual respiratory hospital discharge counts
(ICD-10 J00–J99) by canton of residence for Cotopaxi Province, 2022 and 2024, directly
from the raw INEC microdata (Registro Estadístico de Camas y Egresos Hospitalarios).

It exists because an earlier version of Table 1 in the manuscript contained a
canton-alignment error: the 2022 discharge totals for five of seven cantons had been
assigned to the wrong canton (likely from a list/`groupby` ordering mismatch in an
earlier notebook). The 2024 column, and the Latacunga/La Maná 2022 values, were
unaffected and are reproduced here unchanged.

**Fix applied in this notebook.** Every aggregation below joins by **canton name**
(`cant_res`, accent-normalized), never by row position or list order, which is the
only way this class of bug can recur.

**Inputs expected** (not included in this repository — request access to the raw
INEC microdata via https://www.ecuadorencifras.gob.ec/camas-y-egresos-hospitalarios/):
- `egresos_hospitalarios_2022.sav`
- `egresos_hospitalarios_2024.csv` (semicolon-separated)

**Output:** `Table1_verified.csv` with the corrected per-canton annual totals, population,
and discharge rate per 10,000 inhabitants for 2022 and 2024.

In [ ]:
# ============================================================
# CELL 1 — setup
# ============================================================
!pip install -q pyreadstat

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import unicodedata

pd.set_option('display.width', 140)

# ----- EDIT THESE PATHS to where your raw INEC files live in Drive -----
PATH_2022 = '/content/drive/MyDrive/Geosalud/egresos_hospitalarios_2022.sav'
PATH_2024 = '/content/drive/MyDrive/Geosalud/egresos_hospitalarios_2024.csv'

print('Ready.')

In [ ]:
# ============================================================
# CELL 2 — helpers
# ============================================================
def strip_accents(s):
    """Normalize canton names so 'Saquisilí'/'SAQUISILI'/'saquisili' all match."""
    return ''.join(
        c for c in unicodedata.normalize('NFD', str(s))
        if unicodedata.category(c) != 'Mn'
    ).upper().strip()

# Canonical canton list used throughout the paper. Values are matched by
# substring on the accent-stripped residence canton (cant_res), which is
# robust to the small spelling variants between the 2022 (.sav, labelled)
# and 2024 (.csv, text) files.
CANTON_KEYS = {
    'LATACUNGA': 'Latacunga',
    'PUJIL':     'Pujilí',
    'SALCEDO':   'Salcedo',
    'LA MANA':   'La Maná',
    'SAQUISIL':  'Saquisilí',
    'PANGUA':    'Pangua',
    'SIGCHOS':   'Sigchos',
}

COLS_NEEDED = ['prov_res', 'cant_res', 'cau_cie10']

def load_cotopaxi_respiratory(path, year, is_csv):
    """Load one year's discharge file, keep Cotopaxi residents only, keep
    ICD-10 chapter J (respiratory) only, and tag each row with its canonical
    canton name via CANTON_KEYS (name-based, never positional).
    """
    if is_csv:
        chunks = []
        for chunk in pd.read_csv(path, sep=';', usecols=COLS_NEEDED,
                                  chunksize=200_000, low_memory=False):
            mask = chunk['prov_res'].astype(str).str.strip().str.upper() == 'COTOPAXI'
            if mask.any():
                chunks.append(chunk[mask])
        df = pd.concat(chunks, ignore_index=True)
    else:
        df_full = pd.read_spss(path, usecols=COLS_NEEDED, convert_categoricals=True)
        df = df_full[df_full['prov_res'].astype(str).str.strip().str.upper() == 'COTOPAXI'].copy()
        del df_full

    code = df['cau_cie10'].astype(str).str.strip().str.upper()
    df = df[code.str.startswith('J')].copy()

    df['canton_norm'] = df['cant_res'].apply(strip_accents)
    df['canton'] = None
    for key, nice_name in CANTON_KEYS.items():
        df.loc[df['canton_norm'].str.contains(key), 'canton'] = nice_name

    unmatched = df['canton'].isna().sum()
    if unmatched:
        print(f'  WARNING: {unmatched} rows did not match any known canton (year {year}).')

    df['year'] = year
    return df.dropna(subset=['canton'])

print('Helpers defined.')

In [ ]:
# ============================================================
# CELL 3 — load both years
# ============================================================
print('Loading 2022 (.sav)...')
df22 = load_cotopaxi_respiratory(PATH_2022, 2022, is_csv=False)
print(f'  {len(df22)} respiratory discharges, Cotopaxi residents, 2022')

print('Loading 2024 (.csv)...')
df24 = load_cotopaxi_respiratory(PATH_2024, 2024, is_csv=True)
print(f'  {len(df24)} respiratory discharges, Cotopaxi residents, 2024')

df_all = pd.concat([df22, df24], ignore_index=True)
df_all.to_csv('/content/egresos_respiratorios_cotopaxi_raw.csv', index=False)
print('Saved raw extract to /content/egresos_respiratorios_cotopaxi_raw.csv')

In [ ]:
# ============================================================
# CELL 4 — build corrected Table 1
# ============================================================
# Population figures as published in the manuscript (Table 1, footnote 2):
# 2022 from INEC census-based figures; 2024 is a geometric-growth projection
# from the 2010-2022 intercensal period. These are independent of the
# discharge-count bug and are kept as originally sourced.
POPULATION = {
    'Latacunga': {2022: 182457, 2024: 183959},
    'Pujilí':    {2022: 69870,  2024: 70152},
    'Salcedo':   {2022: 58328,  2024: 58328},
    'La Maná':   {2022: 45635,  2024: 45900},
    'Saquisilí': {2022: 25320,  2024: 25395},
    'Pangua':    {2022: 21969,  2024: 21969},
    'Sigchos':   {2022: 21408,  2024: 21408},
}

CANTON_ORDER = ['Latacunga', 'Pujilí', 'Salcedo', 'La Maná', 'Saquisilí', 'Pangua', 'Sigchos']

counts = df_all.groupby(['canton', 'year']).size().unstack(fill_value=0)

rows = []
for canton in CANTON_ORDER:
    n22 = int(counts.loc[canton, 2022]) if 2022 in counts.columns else 0
    n24 = int(counts.loc[canton, 2024]) if 2024 in counts.columns else 0
    pop22 = POPULATION[canton][2022]
    pop24 = POPULATION[canton][2024]
    rate22 = n22 / pop22 * 10000
    rate24 = n24 / pop24 * 10000
    rows.append({
        'Canton': canton,
        'Population_2022': pop22, 'Population_2024': pop24,
        'Discharges_2022': n22, 'Discharges_2024': n24,
        'Rate_per_10k_2022': round(rate22, 1),
        'Rate_per_10k_2024': round(rate24, 1),
        'Rate_change_pct': round((rate24 - rate22) / rate22 * 100, 1),
    })

table1 = pd.DataFrame(rows)
total_row = {
    'Canton': 'Total',
    'Population_2022': table1['Population_2022'].sum(),
    'Population_2024': table1['Population_2024'].sum(),
    'Discharges_2022': table1['Discharges_2022'].sum(),
    'Discharges_2024': table1['Discharges_2024'].sum(),
    'Rate_per_10k_2022': None, 'Rate_per_10k_2024': None, 'Rate_change_pct': None,
}
table1 = pd.concat([table1, pd.DataFrame([total_row])], ignore_index=True)

print(table1.to_string(index=False))
table1.to_csv('/content/Table1_verified.csv', index=False)

from google.colab import files
files.download('/content/Table1_verified.csv')

## Expected result

| Canton | Discharges 2022 | Discharges 2024 |
|---|---|---|
| Latacunga | 1,217 | 1,418 |
| Pujilí | 278 | 290 |
| Salcedo | 307 | 273 |
| La Maná | 110 | 73 |
| Saquisilí | 70 | 78 |
| Pangua | 66 | 40 |
| Sigchos | 87 | 61 |
| **Total** | **2,135** | **2,233** |

These match the corrected Table 1 published in the manuscript. The 2022 total (2,135)
is unchanged from the original (pre-correction) manuscript — only its distribution
across cantons was wrong.

**Scope note.** This notebook verifies Table 1 (annual totals and rates) only. The
weekly SO₂–discharge correlation analysis (Table 2/3, Poisson GLM, and the
weeks 40–52 eruptive-period comparison) is **not** reproduced here; those figures
remain as originally published pending access to the original analysis notebook.